In [ ]:
# ── INSTALAÇÃO AUTOMÁTICA DE DEPENDÊNCIAS ────────────────────────────────────
# Esta célula garante que todas as bibliotecas necessárias estão instaladas.
# Rode ela primeiro — ela detecta o que falta e instala automaticamente.

import subprocess, sys, importlib

PACOTES = {
    'pandas':      'pandas>=2.0',
    'numpy':       'numpy>=1.26',
    'matplotlib':  'matplotlib>=3.8',
    'yfinance':    'yfinance>=0.2.40',
    'pyarrow':     'pyarrow>=15.0',
    'scipy':       'scipy>=1.11',
    'statsmodels': 'statsmodels>=0.14',
    'sklearn':     'scikit-learn>=1.4',
    'anthropic':   'anthropic>=0.25',
}

instalou = False
for modulo, pacote in PACOTES.items():
    try:
        importlib.import_module(modulo)
    except ImportError:
        print(f"Instalando {pacote}...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pacote],
                       check=False)
        instalou = True

if instalou:
    print("\n✓ Instalação concluída.")
    print("  Reiniciando o kernel para carregar as novas bibliotecas...")
    # Reinicia o kernel automaticamente (funciona no VS Code e no Colab)
    try:
        import IPython
        IPython.Application.instance().kernel.do_shutdown(restart=True)
    except Exception:
        print("  Reinicie o kernel manualmente (Ctrl+Shift+P → Restart Kernel)")
        print("  e rode todas as células novamente.")
else:
    print("✓ Todas as dependências já estão instaladas. Pode continuar.")

# Aula 5 — Backtest v1: Métricas de Desempenho

**Intensivão Quant AI — ImpactUFSCar**

---

Na aula anterior você construiu a carteira. Hoje você vai medir se ela é boa — e o que "boa" significa em finanças quantitativas.

Ao final desta aula, você terá:
- Entendido Sharpe ratio, drawdown e alpha/beta
- Uma função `calcular_metricas()` reutilizável
- O tear sheet completo da estratégia v1 vs IBOVESPA

In [ ]:
# ── CONFIGURAÇÃO DO AMBIENTE ─────────────────────────────────────────────────
# Detecta se está rodando no VS Code ou no Google Colab e define DADOS_DIR —
# a pasta onde todos os arquivos de dados do curso serão salvos.
# Funciona em qualquer computador, sem precisar alterar nada.

import os, sys, subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # ── Google Colab ──────────────────────────────────────────────────────────
    print("Ambiente: Google Colab")
    from google.colab import drive
    drive.mount('/content/drive')

    # Clonar o repositório do curso se ainda não existir
    REPO_DIR = '/content/intensivao-quant-ai'
    if not os.path.exists(REPO_DIR):
        print("Clonando repositório do curso...")
        subprocess.run(['git', 'clone',
                        'https://github.com/fmaldonadoo/intensivao-quant-ai.git',
                        REPO_DIR], check=False)

    # Dados salvos no Google Drive — persistem entre sessões
    DADOS_DIR = '/content/drive/MyDrive/intensivao_quant/dados'

else:
    # ── VS Code / Jupyter local ───────────────────────────────────────────────
    print("Ambiente: VS Code / Jupyter local")

    # Sobe pastas até encontrar a raiz do repositório (onde fica o .git)
    # Funciona independente de onde o usuário salvou o projeto
    _p = os.path.abspath(os.getcwd())
    _root = None
    while _p != os.path.dirname(_p):
        if os.path.exists(os.path.join(_p, '.git')):
            _root = _p
            break
        _p = os.path.dirname(_p)

    if _root is None:
        # Fallback: usa a pasta onde o notebook está
        _root = os.path.dirname(os.path.abspath('__file__'))

    DADOS_DIR = os.path.join(_root, 'dados')

os.makedirs(DADOS_DIR, exist_ok=True)
print(f"Pasta de dados: {DADOS_DIR}")

In [ ]:
# ── VERIFICAÇÃO DE DADOS DA AULA ANTERIOR ────────────────────────────────────
_necessarios = [
    os.path.join(DADOS_DIR, 'retornos_mensais_limpo.parquet'),
    os.path.join(DADOS_DIR, 'sinal_v1.parquet'),
]
_faltando = [f for f in _necessarios if not os.path.exists(f)]
if _faltando:
    print("\n⚠  Arquivos necessários não encontrados:")
    for f in _faltando:
        print(f"   • {os.path.basename(f)}")
    print(f"\n   Execute primeiro: aula-04-sinal-v1")
    print(f"   Dados esperados em: {DADOS_DIR}")
else:
    print("✓  Dados encontrados. Pode continuar.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Carregando os dados das aulas anteriores
ret_mensais  = pd.read_parquet(os.path.join(DADOS_DIR, 'retornos_mensais_limpo.parquet'))
pesos        = pd.read_parquet(os.path.join(DADOS_DIR, 'pesos_v1.parquet'))
ret_carteira = pd.read_parquet(os.path.join(DADOS_DIR, 'retorno_carteira_v1.parquet'))['ret_momentum']

# Benchmark: IBOVESPA
ibov_raw  = yf.download('^BVSP', start='2010-01-01', end='2024-12-31',
                         auto_adjust=True, progress=False)['Close'].squeeze()
ret_ibov  = np.log(ibov_raw / ibov_raw.shift(1)).resample('ME').sum()
ret_ibov  = ret_ibov.reindex(ret_carteira.index).dropna()
ret_carteira = ret_carteira.reindex(ret_ibov.index).dropna()

print(f"Período do backtest: {ret_carteira.index[0].date()} a {ret_carteira.index[-1].date()}")
print(f"Total de meses:      {len(ret_carteira)}")

In [ ]:
# ── VERIFICAÇÃO DE DEPENDÊNCIAS ──────────────────────────────────────────────
# Este notebook depende dos dados gerados pela aula anterior.
# Se algum arquivo estiver faltando, rode o notebook indicado primeiro.

_arquivos_necessarios = [
    os.path.join(DADOS_DIR, 'retornos_mensais_limpo.parquet'),
    os.path.join(DADOS_DIR, 'sinal_v1.parquet'),
]

_faltando = [f for f in _arquivos_necessarios if not os.path.exists(f)]
if _faltando:
    print("\n⚠  ATENÇÃO: arquivos necessários não encontrados:")
    for f in _faltando:
        print(f"   Faltando: {os.path.basename(f)}")
    print(f"\n   Execute primeiro o notebook: aula-04-sinal-v1")
    print(f"   Os dados devem ficar em: {DADOS_DIR}")
else:
    print("✓  Todos os arquivos necessários encontrados.")

## 1. Retorno Acumulado e a Taxa de Crescimento Anual Composta (CAGR)

Ao avaliar a performance histórica de qualquer estratégia de investimento, o retorno total absoluto acumulado no período de teste é a métrica mais intuitiva, mas ela é altamente incompleta se ignorarmos o fator tempo. Para tornar estratégias testadas em períodos de durações diferentes comparáveis, nós calculamos o **CAGR (Compound Annual Growth Rate)**.

---

### A. Retorno Acumulado Total ($R_{\text{cum}}$)
O retorno acumulado representa o ganho total que um investidor teria se tivesse colocado R$ 1.00 no início do backtest e mantido todo o dinheiro investido na estratégia até o final.

$$R_{\text{cum}} = \prod_{t=1}^T (1 + R_p,t) - 1$$

---

### B. CAGR (Taxa de Crescimento Anual Composta)
O **CAGR** calcula a taxa de retorno geométrico equivalente que, se tivesse sido constante ano após ano ao longo de todo o período, teria gerado exatamente o mesmo retorno acumulado final.

A fórmula geral para dados mensais é:

$$\text{CAGR} = (1 + R_{\text{cum}})^{\frac{12}{M}} - 1$$

Onde:
- $R_{\text{cum}}$ é o retorno acumulado total da estratégia (em decimal).
- $M$ é o número total de **meses** de duração do backtest.

**Intuição para Iniciantes:** Dizer que um fundo rendeu $100\%$ de retorno acumulado em 10 anos parece impressionante. Mas ao calcular o CAGR, descobrimos que isso equivale a um crescimento geométrico anualizado de apenas $\approx 7.18\%$. O CAGR é a métrica padrão que remove a distorção do tempo.


In [ ]:
# Retorno acumulado
acum_mom  = ret_carteira.cumsum()
acum_ibov = ret_ibov.cumsum()

# Retorno total em %
ret_total_mom  = np.exp(acum_mom.iloc[-1])  - 1
ret_total_ibov = np.exp(acum_ibov.iloc[-1]) - 1

# CAGR
n_anos = len(ret_carteira) / 12
cagr_mom  = (1 + ret_total_mom)  ** (1 / n_anos) - 1
cagr_ibov = (1 + ret_total_ibov) ** (1 / n_anos) - 1

print(f"Período: {n_anos:.1f} anos")
print(f"{'':25} {'Momentum':>12} {'IBOVESPA':>12}")
print(f"{'Retorno total':25} {ret_total_mom:>12.1%} {ret_total_ibov:>12.1%}")
print(f"{'CAGR':25} {cagr_mom:>12.1%} {cagr_ibov:>12.1%}")

## 2. O Sharpe Ratio e a Métrica de Risco Anualizada

O **Sharpe Ratio** (criado pelo Nobel de Economia William F. Sharpe em 1966) é a métrica mais famosa e amplamente utilizada em toda a indústria global de gestão de recursos para responder a uma pergunta simples: **o retorno excedente gerado por este fundo vale o risco que ele correu?**

Não adianta ter uma estratégia com retorno altíssimo se ela oscila de forma insustentável no dia a dia, correndo o risco de quebrar o investidor.

---

### A Equação Matemática do Sharpe Ratio

$$\text{Sharpe} = \frac{\mathbb{E}[R_p - R_f]}{\sigma_p}$$

Onde:
- $\mathbb{E}[R_p - R_f]$ é a média do retorno excedente da carteira (retorno da carteira menos a taxa livre de risco, como o CDI no Brasil).
- $\sigma_p$ é o desvio padrão (volatilidade) dos retornos excedentes da carteira.

---

### Como Anualizar o Sharpe Ratio? (Regra da Raiz Quadrada do Tempo)
Normalmente, calculamos a média e o desvio padrão usando a frequência dos nossos dados de backtest (neste caso, retornos mensais). Para tornar os resultados compreensíveis internacionalmente, nós anualizamos a métrica.

Como a variância escala linearmente com o tempo, a volatilidade (desvio padrão) escala com a **raiz quadrada do tempo**.
Para dados mensais, a fórmula de anualização é:

$$\text{Sharpe}_{\text{anualizado}} = \frac{\text{Média Mensal de } (R_p - R_f)}{\text{Desvio Padrão Mensal de } R_p} \times \sqrt{12}$$

### O que constitui um bom Sharpe Ratio no mundo real?
- **Sharpe < 0.0:** A estratégia rende menos que a taxa livre de risco (péssimo).
- **0.0 < Sharpe < 0.5:** A estratégia gera algum alfa, mas com muita volatilidade. Pouco eficiente.
- **0.5 <= Sharpe < 1.0:** Retorno muito bom pelo risco corrido. É o patamar padrão de excelentes fundos de ações de gestão ativa.
- **Sharpe >= 1.0:** Excelente! O retorno excedente compensa perfeitamente a volatilidade.


In [ ]:
def sharpe_anualizado(retornos_mensais, rf_mensal=0.0):
    excesso = retornos_mensais - rf_mensal
    return (excesso.mean() / excesso.std()) * np.sqrt(12)

sharpe_mom  = sharpe_anualizado(ret_carteira)
sharpe_ibov = sharpe_anualizado(ret_ibov)

print(f"Sharpe anualizado — Momentum: {sharpe_mom:.2f}")
print(f"Sharpe anualizado — IBOVESPA: {sharpe_ibov:.2f}")

## 3. Drawdown e Max Drawdown (A Dor Máxima do Investidor)

Embora a volatilidade ($\sigma$) seja a métrica estatística padrão de risco, ela não captura perfeitamente a dor psicológica real que um investidor sente no mundo real. Para isso, usamos o **Drawdown**.

---

### O que é o Drawdown?
O **Drawdown** mede a queda percentual temporária que a carteira de um investidor sofre a partir do seu ponto de valor máximo histórico (pico) até atingir o ponto mais baixo da queda (vale).

A fórmula do drawdown em cada instante $t$ é:

$$\text{Drawdown}_t = \frac{\text{Pico}_t - \text{Valor}_t}{\text{Pico}_t}$$

Onde:
- $\text{Pico}_t = \max_{\tau \le t} (V_{\tau})$ é o maior valor histórico atingido pela carteira até o momento $t$.
- $V_t$ é o valor da carteira no instante $t$.

---

### Max Drawdown (Drawdown Máximo — $MDD$)
O **Max Drawdown** é simplesmente a pior queda pico-a-vale registrada ao longo de toda a história do backtest:

$$\text{Max Drawdown} = \max_{t} (\text{Drawdown}_t)$$

**Intuição Prática e Aspecto Psicológico:**
Imagine que você tem R$ 10.000 investidos. Seu patrimônio sobe para R$ 15.000 (novo pico), depois cai para R$ 9.000 (vale), e depois se recupera até R$ 20.000.
O seu drawdown máximo foi de:
$$\text{MDD} = \frac{15.000 - 9.000}{15.000} = 40\%$$

Se o seu backtest mostra um retorno acumulado de $500\%$, mas exibe um **Max Drawdown de $60\%$**, a estratégia é altamente instável. Na prática, quase nenhum investidor ou cliente aguenta ver $60\%$ do seu capital evaporar temporariamente sem resgatar o dinheiro em pânico e demitir o gestor.
Consequentemente, a indústria quant prioriza estratégias que maximizam o retorno enquanto mantêm o Drawdown Máximo sob estrito controle.


In [ ]:
def calcular_drawdown(retornos_mensais):
    """Retorna série de drawdown e o max drawdown."""
    # Valor da carteira (base 1.0)
    valor = np.exp(retornos_mensais.cumsum())
    # Pico acumulado até cada ponto
    pico = valor.cummax()
    # Drawdown = queda percentual em relação ao pico
    drawdown = (valor - pico) / pico
    return drawdown, drawdown.min()

dd_mom,  mdd_mom  = calcular_drawdown(ret_carteira)
dd_ibov, mdd_ibov = calcular_drawdown(ret_ibov)

print(f"Max Drawdown — Momentum: {mdd_mom:.1%}")
print(f"Max Drawdown — IBOVESPA: {mdd_ibov:.1%}")

# Data do max drawdown
print(f"\nData do pior drawdown da estratégia: {dd_mom.idxmin().strftime('%b/%Y')}")

## 4. Alpha ($\alpha$) e Beta ($\beta$) de Sharpe (O Modelo CAPM)

Toda a teoria clássica de precificação de ativos baseia-se na separação do retorno de uma carteira em dois componentes: o retorno que vem do comportamento geral do mercado amplo (risco sistemático) e o retorno que vem da habilidade específica de seleção do gestor (retorno ativo). Essa relação é modelada pela famosa regressão do **CAPM (Capital Asset Pricing Model)**:

$$R_{p,t} - R_{f,t} = \alpha + \beta (R_{m,t} - R_{f,t}) + \epsilon_t$$

Onde:
- $R_{p,t}$ é o retorno do portfólio no período $t$.
- $R_{f,t}$ é o retorno da taxa livre de risco no período $t$.
- $R_{m,t}$ é o retorno do benchmark de mercado (ex: índice IBOVESPA) no período $t$.
- $\epsilon_t$ é o termo de erro aleatório da regressão (ruído).

---

### A. O Beta ($\beta$ — Risco Sistemático)
O **Beta** mede a sensibilidade ou o risco sistemático do portfólio em relação ao mercado de ações como um todo. Ele descreve a inclinação da reta da regressão linear.
- **$\beta = 1.0$:** A carteira anda perfeitamente em linha com o mercado. Se o IBOVESPA subir $10\%$, a carteira deve subir $10\%$.
- **$\beta > 1.0$:** A carteira é mais sensível (agressiva). Um beta de $1.2$ indica que se o mercado subir $10\%$, a carteira tende a subir $12\%$, mas se cair $10\%$, a carteira cai $12\%$.
- **$\beta < 1.0$:** A carteira é defensiva. Um beta de $0.7$ indica menor sensibilidade a oscilações do mercado.

---

### B. O Alpha ($\alpha$ — Geração de Valor Ativo)
O **Alpha** representa o intercepto da regressão. Ele é a medida definitiva de geração de valor excedente que **não pode ser explicada pelas oscilações gerais do mercado**.
- Um **Alpha positivo significante** indica que a estratégia de seleção sistemática (neste caso, a seleção de momentum) foi capaz de gerar retornos adicionais consistentes que superam a compensação pura de apenas tomar risco de mercado amplo. É a "métrica sagrada" que todo gestor de fundos busca obter.


In [ ]:
# Regressão: ret_carteira = alpha + beta * ret_ibov
slope, intercept, r_value, p_value, std_err = stats.linregress(
    ret_ibov, ret_carteira
)

beta  = slope
alpha = intercept * 12  # anualizado

print(f"Beta:           {beta:.2f}")
print(f"Alpha anualizado: {alpha:.2%}")
print(f"R²:             {r_value**2:.2f}  (quanto do retorno é explicado pelo mercado)")

# Visualização
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(ret_ibov, ret_carteira, alpha=0.4, s=20, label='meses')

x_line = np.linspace(ret_ibov.min(), ret_ibov.max(), 100)
ax.plot(x_line, intercept + slope * x_line,
        color='red', linewidth=2,
        label=f'α={alpha:.1%}/ano  β={beta:.2f}  R²={r_value**2:.2f}')

ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('Retorno IBOVESPA (mensal)')
ax.set_ylabel('Retorno Momentum (mensal)')
ax.set_title('Regressão — Momentum vs IBOVESPA')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Função `calcular_metricas()` — reutilizável

Vamos empacotar todas as métricas em uma função. Você vai usar isso em todas as aulas seguintes para comparar versões da estratégia.

In [ ]:
def calcular_metricas(retornos, benchmark=None, nome='Estratégia', rf_mensal=0.0):
    """
    Calcula métricas de desempenho para uma série de retornos mensais (log).
    
    Parâmetros
    ----------
    retornos    : pd.Series — retornos log mensais da estratégia
    benchmark   : pd.Series — retornos log mensais do benchmark (opcional)
    nome        : str       — nome da estratégia para exibição
    rf_mensal   : float     — taxa livre de risco mensal (default 0)
    
    Retorna
    -------
    dict com todas as métricas
    """
    n_meses = len(retornos)
    n_anos  = n_meses / 12

    # Retorno total e CAGR
    ret_total = np.exp(retornos.sum()) - 1
    cagr      = (1 + ret_total) ** (1 / n_anos) - 1

    # Volatilidade
    vol_anual = retornos.std() * np.sqrt(12)

    # Sharpe
    excesso = retornos - rf_mensal
    sharpe  = (excesso.mean() / excesso.std()) * np.sqrt(12)

    # Sortino (penaliza apenas volatilidade negativa)
    ret_neg    = excesso[excesso < 0]
    vol_neg    = ret_neg.std() * np.sqrt(12)
    sortino    = (excesso.mean() * 12) / vol_neg if vol_neg > 0 else np.nan

    # Drawdown
    valor     = np.exp(retornos.cumsum())
    pico      = valor.cummax()
    drawdown  = (valor - pico) / pico
    mdd       = drawdown.min()

    # Calmar ratio (CAGR / |Max Drawdown|)
    calmar = cagr / abs(mdd) if mdd != 0 else np.nan

    # Win rate (% de meses positivos)
    win_rate = (retornos > rf_mensal).mean()

    # Alpha e Beta vs benchmark
    alpha, beta_val = np.nan, np.nan
    if benchmark is not None:
        idx_comuns = retornos.index.intersection(benchmark.index)
        if len(idx_comuns) > 10:
            sl, ic, _, _, _ = stats.linregress(
                benchmark[idx_comuns], retornos[idx_comuns]
            )
            beta_val = sl
            alpha    = ic * 12

    return {
        'nome':        nome,
        'retorno_total': ret_total,
        'cagr':        cagr,
        'vol_anual':   vol_anual,
        'sharpe':      sharpe,
        'sortino':     sortino,
        'max_drawdown': mdd,
        'calmar':      calmar,
        'win_rate':    win_rate,
        'alpha_anual': alpha,
        'beta':        beta_val,
        'n_meses':     n_meses,
    }


def exibir_metricas(*dicts_metricas):
    """Exibe várias estratégias lado a lado."""
    formatos = {
        'retorno_total': '{:.1%}',
        'cagr':          '{:.1%}',
        'vol_anual':     '{:.1%}',
        'sharpe':        '{:.2f}',
        'sortino':       '{:.2f}',
        'max_drawdown':  '{:.1%}',
        'calmar':        '{:.2f}',
        'win_rate':      '{:.1%}',
        'alpha_anual':   '{:.2%}',
        'beta':          '{:.2f}',
        'n_meses':       '{:.0f}',
    }
    labels = {
        'retorno_total': 'Retorno Total',
        'cagr':          'CAGR',
        'vol_anual':     'Volatilidade Anual',
        'sharpe':        'Sharpe',
        'sortino':       'Sortino',
        'max_drawdown':  'Max Drawdown',
        'calmar':        'Calmar',
        'win_rate':      'Win Rate',
        'alpha_anual':   'Alpha (anual)',
        'beta':          'Beta',
        'n_meses':       'Meses',
    }
    nomes = [d['nome'] for d in dicts_metricas]
    col_w = max(14, max(len(n) for n in nomes) + 2)

    header = f"{'Métrica':<22}" + ''.join(f"{n:>{col_w}}" for n in nomes)
    print(header)
    print('─' * len(header))
    for chave, label in labels.items():
        linha = f"{label:<22}"
        for d in dicts_metricas:
            v = d.get(chave, np.nan)
            s = formatos[chave].format(v) if not (isinstance(v, float) and np.isnan(v)) else 'N/A'
            linha += f"{s:>{col_w}}"
        print(linha)


m_mom  = calcular_metricas(ret_carteira, benchmark=ret_ibov, nome='Momentum v1')
m_ibov = calcular_metricas(ret_ibov, nome='IBOVESPA')

print()
exibir_metricas(m_mom, m_ibov)

## 6. Tear sheet — o relatório visual da estratégia

Um tear sheet é a visualização padronizada que gestores profissionais usam para avaliar uma estratégia. Três painéis essenciais: retorno acumulado, drawdown, e retornos mensais.

In [ ]:
def plot_tearsheet(ret_estrategia, ret_benchmark, nome_estrategia='Estratégia'):
    fig = plt.figure(figsize=(14, 11))
    gs  = gridspec.GridSpec(3, 1, height_ratios=[3, 1.5, 1.5], hspace=0.35)

    # ── Painel 1: Retorno acumulado ──────────────────────────────────────
    ax1 = fig.add_subplot(gs[0])

    valor_est = np.exp(ret_estrategia.cumsum())
    valor_bm  = np.exp(ret_benchmark.reindex(ret_estrategia.index).cumsum())

    ax1.plot(valor_est,  label=nome_estrategia, linewidth=2, color='steelblue')
    ax1.plot(valor_bm,   label='IBOVESPA',       linewidth=1.5,
             linestyle='--', color='gray')
    ax1.set_ylabel('Valor da carteira (base 1.0)')
    ax1.set_title(f'Tear Sheet — {nome_estrategia}', fontsize=13, fontweight='bold')
    ax1.legend(loc='upper left')

    # Anotação das métricas no canto
    m = calcular_metricas(ret_estrategia, ret_benchmark, nome_estrategia)
    texto = (f"CAGR: {m['cagr']:.1%}   Vol: {m['vol_anual']:.1%}\n"
             f"Sharpe: {m['sharpe']:.2f}   MDD: {m['max_drawdown']:.1%}\n"
             f"Alpha: {m['alpha_anual']:.2%}   Beta: {m['beta']:.2f}")
    ax1.text(0.02, 0.97, texto, transform=ax1.transAxes,
             verticalalignment='top', fontsize=9,
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    # ── Painel 2: Drawdown ───────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1], sharex=ax1)

    valor_bm_reindexado = np.exp(ret_benchmark.reindex(ret_estrategia.index).fillna(0).cumsum())
    pico_est = valor_est.cummax()
    pico_bm  = valor_bm_reindexado.cummax()
    dd_est   = (valor_est - pico_est) / pico_est
    dd_bm    = (valor_bm_reindexado - pico_bm) / pico_bm

    ax2.fill_between(dd_est.index, dd_est, 0, alpha=0.4, color='steelblue', label=nome_estrategia)
    ax2.plot(dd_bm, linewidth=1, color='gray', linestyle='--', label='IBOVESPA', alpha=0.7)
    ax2.set_ylabel('Drawdown')
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax2.legend(loc='lower left', fontsize=8)

    # ── Painel 3: Retornos mensais ───────────────────────────────────────
    ax3 = fig.add_subplot(gs[2], sharex=ax1)

    cores = ['steelblue' if r >= 0 else 'tomato' for r in ret_estrategia]
    ax3.bar(ret_estrategia.index, ret_estrategia, width=20,
            color=cores, alpha=0.8)
    ax3.axhline(0, color='black', linewidth=0.6)
    ax3.set_ylabel('Retorno mensal')
    ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

    plt.tight_layout()
    plt.savefig('tearsheet_v1.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Salvo: tearsheet_v1.png")


plot_tearsheet(ret_carteira, ret_ibov, nome_estrategia='Momentum v1 (equal weight, top 10)')

## 7. Retornos mensais por ano — heatmap

Outra visualização padrão de relatórios profissionais: a tabela de retornos mensais por ano. Permite identificar em que períodos a estratégia funciona e quando não funciona.

In [ ]:
# Tabela de retornos mensais × anos
df_ret = ret_carteira.to_frame('retorno')
df_ret['ano'] = df_ret.index.year
df_ret['mes'] = df_ret.index.month

meses_nome = {1:'Jan',2:'Fev',3:'Mar',4:'Abr',5:'Mai',6:'Jun',
              7:'Jul',8:'Ago',9:'Set',10:'Out',11:'Nov',12:'Dez'}

tabela = df_ret.pivot(index='ano', columns='mes', values='retorno')
tabela.columns = [meses_nome[m] for m in tabela.columns]
tabela['Total'] = df_ret.groupby('ano')['retorno'].sum().apply(lambda x: np.exp(x) - 1)

fig, ax = plt.subplots(figsize=(14, 6))
import matplotlib.colors as mcolors
cmap = mcolors.LinearSegmentedColormap.from_list('rg', ['tomato', 'white', 'steelblue'])

im = ax.imshow(tabela.values, cmap=cmap, aspect='auto',
               vmin=-0.15, vmax=0.15)

ax.set_xticks(range(len(tabela.columns)))
ax.set_xticklabels(tabela.columns)
ax.set_yticks(range(len(tabela.index)))
ax.set_yticklabels(tabela.index)

for i in range(len(tabela.index)):
    for j in range(len(tabela.columns)):
        v = tabela.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:.1%}', ha='center', va='center', fontsize=7.5,
                    color='black' if abs(v) < 0.10 else 'white')

ax.set_title('Retornos mensais — Momentum v1')
plt.colorbar(im, ax=ax, fraction=0.02)
plt.tight_layout()
plt.show()

## Resumo da aula

| Métrica | O que mede | Fórmula chave |
|---|---|---|
| CAGR | Crescimento anual composto | $(1 + R_{total})^{1/T} - 1$ |
| Sharpe | Retorno por unidade de risco | $(\bar{r} / \sigma) \times \sqrt{12}$ |
| Sortino | Sharpe penalizando só volatilidade negativa | $(\bar{r} / \sigma_{neg}) \times \sqrt{12}$ |
| Max Drawdown | Pior queda de pico a vale | $\min_t \frac{V_t - \max_{s\leq t} V_s}{\max_{s\leq t} V_s}$ |
| Calmar | CAGR relativo ao max drawdown | $\text{CAGR} / |\text{MDD}|$ |
| Alpha | Retorno além do risco de mercado | Intercepto da regressão × 12 |
| Beta | Sensibilidade ao mercado | Coeficiente da regressão |

---

## Para replicar em casa

1. Rode o notebook e confirme que o `tearsheet_v1.png` foi salvo
2. Use `calcular_metricas()` e `exibir_metricas()` para comparar `N_ATIVOS = 5` vs `N_ATIVOS = 10` vs `N_ATIVOS = 15` — qual tem melhor Sharpe?
3. Observe o heatmap: em quais anos a estratégia claramente funcionou? Em quais falhou?

---

*ImpactUFSCar — Diretoria de Quant*